# ハンズオン3: 新素材や薬の研究と量子コンピュータ

## なぜこの分野が注目されるのか

量子コンピュータの応用先として、**最も期待されているのが材料と化学の分野**です。理由を順に見ていきましょう。

### やりたいこと

長持ちする電池、効率の良い触媒、新しい薬。こうしたものを、実験室で試す前に**コンピュータの中で設計したい**というニーズがあります。実験には時間もお金もかかるからです。

そのためには、物質の中で**電子がどう振る舞うか**を計算する必要があります。物質の性質は、結局のところ電子の振る舞いで決まるからです。

### なぜ普通のコンピュータでは難しいのか

電子は**量子の法則**に従って動きます。電子が増えると、考えるべき組み合わせが爆発的に増えていきます。

前回見た「組合せ爆発」と同じことが、ここでも起こります。電子が50個あるような、ごく小さな分子でさえ、正確に計算しようとすると世界最高のスーパーコンピュータでも手に負えません。

### 発想の転換

ここで自然な問いが生まれます。

> **量子の法則で動くものを計算するなら、同じく量子の法則で動く機械を使うほうが自然ではないか？**

これが、物理学者リチャード・ファインマンが1980年代に提案した考え方で、量子コンピュータという発想の原点のひとつです。

## 今回やること

本物の分子は複雑すぎるので、**極限まで単純にしたモデル**を使います。そのうえで「いちばん安定な状態」を探す練習をします。使う手法は **VQE**（ブイキューイー）と呼ばれるものです。

## 覚えておきたい言葉

| 言葉 | 日常語で言うと |
|---|---|
| エネルギーが低い | **安定している**。ボールは坂を転がって谷底で止まります |
| 基底状態 | **いちばん安定な状態**。谷のいちばん底 |
| ハミルトニアン | **エネルギーの計算ルール**。「この状態なら、エネルギーはいくつ」を決める表のようなもの |

化学の授業で「安定な状態」という言葉を聞いたことがあるかもしれません。それは**エネルギーが低い状態**のことです。

## 注意

今回のモデルは教材用の簡易版です。実際の材料の性質を予測できるものではありません。

In [ ]:
import math
import random

# H = c0 I + cx X + cz Z
c0 = -0.80
cx = 0.35
cz = -0.60
SEED = 42

## 1. 先に「正解」を計算しておく

### なぜ正解を先に出すのか

今回のモデルはとても小さいので、紙と鉛筆の数学だけで**厳密な答え**が出せます。

わざわざ正解が分かっている問題を選ぶのには理由があります。これから試す「量子的なやり方」が、**本当に正しい答えにたどり着けるか検証できる**からです。

新しい道具の精度を確かめるときは、答えの分かっているもので試す。体重計を買ったら、まず重さの分かっている物を載せてみるのと同じです。

### 2つのエネルギー

次のセルでは2つの値を計算します。

- **基底状態のエネルギー** … いちばん低い値。谷の底。今回探したいのはこちら
- **励起状態のエネルギー** … それより高い値。不安定な状態

計算式の `±` の部分が、この2つの違いを生んでいます。マイナスを選べば低いほう、プラスを選べば高いほうです。

### 式の中身について

`c0`、`cx`、`cz` という3つの数字が、この物質の性質を決めています。本物の分子では、この部分が何千、何万という数になります。だからこそ難しいのです。

今回は3つだけなので、手計算できます。

In [ ]:
exact_ground_energy = c0 - math.sqrt(cx ** 2 + cz ** 2)
exact_excited_energy = c0 + math.sqrt(cx ** 2 + cz ** 2)
print(f"exact ground energy : {exact_ground_energy:.8f}")
print(f"exact excited energy: {exact_excited_energy:.8f}")

## 2. つまみを回して、谷底を探す（変分法）

### イメージをつかむ

目隠しをして、広い谷のどこかに立っているとします。いちばん低い場所に行きたい。手元には**つまみが1つ**あり、回すと立つ位置が変わります。高度計も持っています。

やることはシンプルです。

1. つまみを少し回す
2. 高度計を見る
3. さっきより低くなったか確認する
4. 繰り返す

これが**変分法**という考え方です。今回のつまみは `θ`（シータ）という角度1つだけです。

### 量子コンピュータでのつまみ

量子コンピュータでは、`Ry(θ)` という回転の操作がつまみにあたります。角度を変えると、量子ビットの状態が少しずつ変わります。

そのときのエネルギーは次の式になります。

`E(θ) = c0 + cx × sin(θ) + cz × cos(θ)`

`sin` と `cos` は高校で習う三角関数です。式が苦手なら「**つまみの角度を決めると、エネルギーが1つ決まる**」とだけ理解すれば先に進めます。

### 変分原理という安心材料

ここで、この手法を支える大切な性質を紹介します。

> **どんな角度を試しても、出てくるエネルギーが「本当の最小値」より低くなることは絶対にない**

これを**変分原理**と呼びます。地面より下には潜れない、というイメージです。

なぜこれが重要かというと、**低い値が出たら、それだけ正解に近づいた**と素直に判断してよいからです。「もしかしたら行きすぎたのでは」と心配する必要がありません。

正解を知らなくても、答えの良し悪しを判断できる。これがVQEという手法が成り立つ理由です。

### 次のセルがすること

角度を0度から360度まで細かく動かし、いちばんエネルギーが低い角度を探します。そして第1節で計算した厳密な正解と比べ、どれくらい近いかを確認します。

誤差は `6.276e-06`（約0.000006）程度になるはずです。非常に小さい値です。

In [ ]:
def energy(theta):
    return c0 + cx * math.sin(theta) + cz * math.cos(theta)

grid = [2 * math.pi * step / 720 for step in range(721)]
best_theta = min(grid, key=energy)
best_energy = energy(best_theta)

print(f"best theta : {best_theta:.6f} rad")
print(f"VQE energy : {best_energy:.8f}")
print(f"error      : {best_energy - exact_ground_energy:.3e}")
assert best_energy >= exact_ground_energy - 1e-12
assert best_energy - exact_ground_energy < 1e-4

## 3. 本物の機械では「平均」しか分からない

### 理想と現実の差

ここまでの計算では、エネルギーの値を直接求めていました。しかし**本物の量子コンピュータは、エネルギーを直接表示してくれません**。

できるのは「測定して `+1` か `-1` のどちらかを得る」ことだけです。そこで何度も測定し、その**平均**を取ることで、知りたい値を推定します。

### アンケートのたとえ

内閣支持率を知りたいとき、国民全員に聞くことはできません。一部の人に聞いて推測します。

- 100人に聞く → だいたいの傾向は分かるが、誤差が大きい
- 10000人に聞く → かなり正確になるが、手間と費用がかかる

量子コンピュータの測定も、まったく同じ構造です。ここで出てくる `shots` が「何人に聞くか」にあたります。

### 次のセルがすること

shotsを `100`、`1000`、`10000` と変えて、推定したエネルギーが本当の値からどれだけズレるかを見ます。

出力の `sampling error` が、そのズレです。shotsを増やすほど小さくなることを確認してください。

In [ ]:
def sample_pauli_expectation(expectation, shots, rng):
    probability_plus = (1.0 + expectation) / 2.0
    outcomes = (1 if rng.random() < probability_plus else -1 for _ in range(shots))
    return sum(outcomes) / shots

def sampled_energy(theta, shots, seed):
    rng = random.Random(seed)
    measured_x = sample_pauli_expectation(math.sin(theta), shots, rng)
    measured_z = sample_pauli_expectation(math.cos(theta), shots, rng)
    return c0 + cx * measured_x + cz * measured_z

for shots in (100, 1_000, 10_000):
    estimate = sampled_energy(best_theta, shots, SEED)
    print(f"shots={shots:>5}: energy={estimate:.6f}, sampling error={estimate - best_energy:+.6f}")

### 結果を見るときの注意

出力をよく見ると、shots が `1000` のときに `-1.510800` という、**厳密な正解 `-1.494622` よりも低い値**が出ています。

さきほど「変分原理があるので、正解より低い値は出ない」と説明したばかりです。矛盾しているように見えますが、違います。

これは**測定回数が少ないために生じた、偶然のズレ**です。1000人のアンケートで、たまたま支持者が多めに選ばれてしまったのと同じことです。真の値が変わったわけではなく、推定の精度が粗いだけです。

### 覚えておくと便利な目安

> **測定回数を4倍にすると、誤差はおよそ半分になる**

精度を2倍にするには4倍のコストがかかる、ということです。実機では実行回数がそのまま料金になるので、「どこまで正確さが必要か」を決めることが、そのまま予算の判断になります。

精度と費用のトレードオフ。ここでも、前回のハンズオンと同じ構造が現れます。

## 4. 量子と古典の「二人三脚」

VQEは、量子コンピュータだけで完結する手法ではありません。**普通のコンピュータと量子コンピュータが交互に作業する**仕組みです。これを**ハイブリッド**と呼びます。

### 役割分担

| 担当 | やること |
|---|---|
| 普通のコンピュータ | 分子のデータを整え、計算ルール（ハミルトニアン）を作る |
| 量子コンピュータ | つまみを設定した状態を作り、測定する |
| 普通のコンピュータ | 測定結果を集計してエネルギーを推定する |
| 普通のコンピュータ | 「次はつまみをこう回そう」と判断する |
| 量子コンピュータ | 新しい設定で、また測定する |

これを答えが落ち着くまで繰り返します。

### 量子コンピュータの出番は意外と少ない

表を見て気づいたと思いますが、**作業の大半は普通のコンピュータが担当しています**。量子コンピュータは、特に難しい一部分だけを受け持つ「専門職」のような存在です。

この構図は、画像処理をGPUに任せる現在のコンピュータの使い方とよく似ています。量子コンピュータは万能の置き換えではなく、**特定の作業に特化した助っ人**だと考えるのが実態に近い理解です。

### 実務上の注意点

量子コンピュータは順番待ちの列があります。結果が返るまで数分から数時間かかることもあります。

このとき、高価な計算機資源を待たせたまま遊ばせておくと、費用だけがかさみます。そのため実際のシステムでは、**待っている間に別の作業ができる**ような作りにすることが重要になります。

## 5. 本物の材料計算との違い

今回は極限まで単純にしたモデルでした。実際の研究では、次のような工程が加わります。

### 計算の前に決めること

- どんな構造の分子・結晶を調べるのか
- どの元素が、どの位置にあるのか
- どこまで細かく電子を表現するか（精度と計算量のバランス）

### 量子コンピュータに渡す前の下準備

- 普通のコンピュータで、おおまかな計算を済ませておく
- **特に難しい部分だけを切り出す**（活性空間の選択と呼びます）
- 電子の性質を、量子ビットの言葉に翻訳する

この「切り出し」が実務では非常に重要です。分子全体を量子コンピュータに載せることは現状できないので、**古典計算が苦手な部分だけを見極めて渡す**必要があります。

### 結果が出たあと

- 実験データと比べて、妥当かどうか検証する
- エネルギーだけでなく、反応の起こりやすさや物性も確認する

### 最初の一歩として現実的なこと

いきなり大きな材料を量子コンピュータで計算しようとしないこと。まずは、

1. 普通のコンピュータでどこまでできるかを把握する
2. 本当に苦手な部分を特定する
3. その部分が今の量子コンピュータで扱える大きさか評価する

という順番で進めるのが現実的です。

## 6. 次に進むなら

### 公式の化学コースを試す

VS Codeに「QDK拡張」を入れると、**Ground-State Molecular Energies with QPE** という公式の学習コースを利用できます。

- 窒素分子という実在の分子を扱います
- QPE（量子位相推定）という、VQEとは別の手法を学びます
- 自分のパソコンの中で動くので、**Azureのアカウントは不要・料金もかかりません**

本教材より一歩進んだ、実践的な内容です。

### 将来の見積もりを立てる

「今の量子コンピュータでは小さすぎる。では、どのくらいの性能になれば実用になるのか」

この問いに答えるための道具が **Resource Estimator**（リソース見積もりツール）です。「この計算をするには、量子ビットが何個、時間がどれだけ必要か」を試算できます。

将来の投資判断や、研究計画を立てるときに使われます。

### リンク

- 化学コース: https://learn.microsoft.com/azure/quantum/qdk-learning-quantum-chemistry
- Resource Estimator: https://learn.microsoft.com/azure/quantum/intro-to-resource-estimation

## 演習

### 演習1: 物質の性質を変えてみる

`cx` と `cz` の値を変えて実行してください。たとえば `cx = 0.70`、`cz = -0.20` などです。

**考えるポイント:** いちばん低いエネルギーを与える角度は、どう変わったでしょうか。この2つの数字は「物質の性質」にあたるので、性質が変われば安定な状態も変わります。

### 演習2: 測定のばらつきを体感する

`SEED` の値を10種類変えて、shots `1000` での結果を集めてください。平均と、ばらつきの大きさを計算してみましょう。

**考えるポイント:** 1回の実験結果だけで判断するのは、なぜ危険なのでしょうか。

### 演習3: 誤差の法則を確かめる

shotsを `250` → `1000` → `4000` と4倍ずつ増やして、誤差がおよそ半分ずつになるか確認してください。

**ヒント:** 複数のSEEDで試して平均を取ると、傾向がはっきり見えます。

### 演習4: 役割分担を考える

あなたの会社や身近な業界で、新しい材料や製品を開発する場面を想像してください。その作業のうち、

- 普通のコンピュータが得意なこと
- AIが得意なこと
- 量子コンピュータに任せたいこと

をそれぞれ書き出してみましょう。

**ヒント:** 量子コンピュータの担当範囲は、思ったより狭くて構いません。